# DatasetPro — Google Colab Runner

Ejecuta el pipeline completo de generación de dataset desde Colab, leyendo
modelos y guardando resultados directamente en Google Drive.

**Estructura esperada en Drive:**
```
MyDrive/
└── datasetpro/
    ├── models/
    │   └── face_parsing/
    │       └── 79999_iter.pth   ← subir manualmente una vez
    └── data/
        └── raw/
            └── real/            ← imágenes FFHQ de entrada
```

## Celda 1 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Celda 2 — Definir rutas (edita `DRIVE_ROOT` si usas otra carpeta)

In [ ]:
from pathlib import Path

# ── Configura aquí la raíz de tu proyecto en Drive ───────────────────────────
DRIVE_ROOT  = "/content/drive/MyDrive/datasetpro"
REPO_DIR    = Path('/content/datasetpro')

MODELS_DIR  = DRIVE_ROOT / 'models'
DATA_DIR    = DRIVE_ROOT / 'data'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODELS_DIR : {MODELS_DIR}')
print(f'DATA_DIR   : {DATA_DIR}')

# Verificar checkpoint de face-parsing
_ckpt = MODELS_DIR / 'face_parsing/79999_iter.pth'
if _ckpt.exists():
    print(f'✓ Checkpoint encontrado: {_ckpt}')
else:
    print(f'⚠ FALTA: {_ckpt}')
    print('  Descárgalo en: https://drive.google.com/open?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812')
    print(f'  y súbelo a Drive en: {_ckpt.parent}/')

## Celda 3 — Clonar o actualizar el repositorio

In [ ]:
import subprocess, os

GITHUB_REPO = 'https://github.com/CuauhtliRojas/datasetpro.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', GITHUB_REPO, str(REPO_DIR)], check=True)
    print('Repositorio clonado.')
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    print('Repositorio actualizado.')

os.chdir(REPO_DIR)
print(f'CWD: {os.getcwd()}')

## Celda 4 — Instalar entorno con `uv`

> La primera ejecución tarda ~5 min; las siguientes son instantáneas
> si el runtime de Colab no se reinició.

In [ ]:
import subprocess

# 1. Instalar uv
subprocess.run(['pip', 'install', 'uv', '--quiet'], check=True)

# 2. Instalar todas las dependencias del proyecto (incluye torch CUDA para Linux
#    gracias al marcador sys_platform=='linux' en pyproject.toml)
subprocess.run(
    ['uv', 'pip', 'install', '-e', '.', '--system', '--quiet'],
    cwd=str(REPO_DIR),
    check=True,
)

print('Entorno listo.')

## Celda 5 — (Opcional) Token de HuggingFace

Necesario si los modelos de `runwayml/stable-diffusion-*` están tras login.
Usa Colab Secrets (llave izquierda) con nombre `HF_TOKEN` en lugar de pegar el token aquí.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('Token de HuggingFace cargado desde Colab Secrets.')
except Exception:
    print('Sin token de HuggingFace — solo modelos públicos disponibles.')

## Celda 6 — Ejecutar scripts

Descomenta el script que quieras correr. Todos leen de `DATA_DIR` y `MODELS_DIR` en Drive.

In [ ]:
import subprocess, sys

def run_script(script: str, extra_args: list[str] | None = None):
    """Ejecuta un script del pipeline pasando las rutas de Drive."""
    cmd = [
        sys.executable, f'scripts/{script}',
        '--data_dir',   str(DATA_DIR),
        '--models_dir', str(MODELS_DIR),
    ] + (extra_args or [])
    print(f'Ejecutando: {" ".join(str(c) for c in cmd)}\n')
    result = subprocess.run(cmd, cwd=str(REPO_DIR), text=True)
    if result.returncode != 0:
        print(f'ERROR (código {result.returncode})')

# ── Elige qué ejecutar ────────────────────────────────────────────────────────

# Paso 2: Face swap (requiere inswapper_128.onnx en ~/.insightface/models/)
# run_script('02_generar_swaps.py')

# Paso 3: Inpainting SD (descarga modelo de HuggingFace ~2 GB)
# run_script('03_generar_inpainting.py')

# Paso 4: Síntesis completa SD (descarga modelo de HuggingFace ~2 GB)
run_script('04_generar_sintesis.py')

# Paso 5: Reenactment FOMM
# run_script('05_generar_reenactment.py')